In [1]:
import os
import sys

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["HADOOP_HOME"] = "/opt/hadoop"
os.environ["HADOOP_CONF_DIR"] = "/opt/hadoop/etc/hadoop"
os.environ["SPARK_HOME"] = "/opt/spark"
os.environ["HIVE_HOME"] = "/opt/hive"

os.environ["PYSPARK_PYTHON"] = "/home/vboxuser/bigdata-hotel-project/venv/bin/python"
os.environ["PYSPARK_DRIVER_PYTHON"] = "/home/vboxuser/bigdata-hotel-project/venv/bin/python"

print(sys.executable)
print(sys.version)

/home/vboxuser/bigdata-hotel-project/venv/bin/python
3.10.20 (main, May  6 2026, 18:14:37) [GCC 15.2.0]


In [2]:
from pathlib import Path
import csv
import math
import os
import re
import sys

from pyspark.ml.feature import VectorAssembler
from pyspark.ml.stat import Correlation
from pyspark.sql import Column, DataFrame, SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T

In [3]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("HotelReservationCancellationClassification")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.memory", "4g") 
    .config("spark.executor.memory", "4g") 
    .config("spark.sql.ansi.enabled", "false")
    .config("spark.sql.warehouse.dir", "hdfs:///user/hive/warehouse")
    .config("spark.sql.catalogImplementation", "hive")
    .config("spark.driver.extraClassPath", "/opt/spark/jars/mariadb-java-client.jar")
    .config("spark.executor.extraClassPath", "/opt/spark/jars/mariadb-java-client.jar")
    .enableHiveSupport()
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/09 12:29:18 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 3.5.1


In [4]:
import csv
import math
import re

from pyspark.ml.feature import VectorAssembler
from pyspark.ml.stat import Correlation
from pyspark.sql import Column, DataFrame
from pyspark.sql import functions as F
from pyspark.sql import types as T

TARGET_COLUMN = "booking_status"
LABEL_COLUMN = "booking_status_label"
RANDOM_SEED = 42

EXPECTED_COLUMNS = [
    "booking_id",
    "no_of_adults",
    "no_of_children",
    "no_of_weekend_nights",
    "no_of_week_nights",
    "type_of_meal_plan",
    "required_car_parking_space",
    "room_type_reserved",
    "lead_time",
    "arrival_year",
    "arrival_month",
    "arrival_date",
    "market_segment_type",
    "repeated_guest",
    "no_of_previous_cancellations",
    "no_of_previous_bookings_not_canceled",
    "avg_price_per_room",
    "no_of_special_requests",
    "booking_status",
]

INTEGER_COLUMNS = [
    "no_of_adults",
    "no_of_children",
    "no_of_weekend_nights",
    "no_of_week_nights",
    "required_car_parking_space",
    "lead_time",
    "arrival_year",
    "arrival_month",
    "arrival_date",
    "repeated_guest",
    "no_of_previous_cancellations",
    "no_of_previous_bookings_not_canceled",
    "no_of_special_requests",
]

NUMERIC_COLUMNS = [*INTEGER_COLUMNS, "avg_price_per_room"]

CATEGORICAL_COLUMNS = [
    "type_of_meal_plan",
    "room_type_reserved",
    "market_segment_type",
    "booking_status",
]

CATEGORICAL_FEATURE_COLUMNS = [
    "type_of_meal_plan",
    "room_type_reserved",
    "market_segment_type",
]

BINARY_COLUMNS = [
    "required_car_parking_space",
    "repeated_guest",
]

VALID_CATEGORY_VALUES = {
    "type_of_meal_plan": ["Meal Plan 1", "Meal Plan 2", "Meal Plan 3", "Not Selected"],
    "room_type_reserved": [
        "Room_Type 1",
        "Room_Type 2",
        "Room_Type 3",
        "Room_Type 4",
        "Room_Type 5",
        "Room_Type 6",
        "Room_Type 7",
    ],
    "market_segment_type": [
        "Aviation",
        "Complementary",
        "Corporate",
        "Offline",
        "Online",
    ],
    "booking_status": ["Canceled", "Not_Canceled"],
}

ONE_HOT_BASELINES = {
    "type_of_meal_plan": "Meal Plan 1",
    "room_type_reserved": "Room_Type 1",
    "market_segment_type": "Online",
}

SCALABLE_FEATURE_COLUMNS = [
    "no_of_adults",
    "no_of_children",
    "no_of_weekend_nights",
    "no_of_week_nights",
    "lead_time",
    "arrival_year",
    "arrival_month",
    "arrival_date",
    "no_of_previous_cancellations",
    "no_of_previous_bookings_not_canceled",
    "avg_price_per_room",
    "no_of_special_requests",
    "total_nights",
    "total_guests",
    "arrival_day_of_week",
    "arrival_quarter",
    "estimated_booking_value",
    "avg_price_per_guest",
    "previous_total_bookings",
    "previous_cancellation_rate",
]

BINARY_FEATURE_COLUMNS = [
    "required_car_parking_space",
    "repeated_guest",
    "has_children",
    "arrival_is_weekend",
]

In [5]:
def validate_schema(df: DataFrame) -> None:
    missing_columns = sorted(set(EXPECTED_COLUMNS) - set(df.columns))
    extra_columns = sorted(set(df.columns) - set(EXPECTED_COLUMNS))

    if missing_columns or extra_columns:
        raise ValueError(
            "Dataset schema mismatch. "
            f"Missing columns: {missing_columns}. Extra columns: {extra_columns}."
        )


def standardize_text_columns(df: DataFrame) -> DataFrame:
    clean = df
    for column in EXPECTED_COLUMNS:
        clean = clean.withColumn(column, F.trim(F.col(column).cast("string")))
    return clean


def coerce_numeric_columns(df: DataFrame) -> DataFrame:
    clean = df
    for column in INTEGER_COLUMNS:
        clean = clean.withColumn(column, F.col(column).cast(T.IntegerType()))

    clean = clean.withColumn("avg_price_per_room", F.col("avg_price_per_room").cast(T.DoubleType()))
    return clean


def add_quality_check_columns(df: DataFrame) -> DataFrame:
    date_string = F.concat_ws(
        "-",
        F.col("arrival_year").cast("string"),
        F.lpad(F.col("arrival_month").cast("string"), 2, "0"),
        F.lpad(F.col("arrival_date").cast("string"), 2, "0"),
    )

    clean = (
        df.withColumn("arrival_date_full", F.to_date(date_string, "yyyy-MM-dd"))
        .withColumn("total_nights_check", F.col("no_of_weekend_nights") + F.col("no_of_week_nights"))
        .withColumn("total_guests_check", F.col("no_of_adults") + F.col("no_of_children"))
    )

    missing_required_condition = None
    for column in EXPECTED_COLUMNS:
        condition = F.col(column).isNull()
        missing_required_condition = condition if missing_required_condition is None else missing_required_condition | condition

    negative_numeric_condition = None
    for column in NUMERIC_COLUMNS:
        condition = F.col(column) < 0
        negative_numeric_condition = condition if negative_numeric_condition is None else negative_numeric_condition | condition

    invalid_category_condition = None
    for column, valid_values in VALID_CATEGORY_VALUES.items():
        condition = F.col(column).isNull() | ~F.col(column).isin(valid_values)
        invalid_category_condition = condition if invalid_category_condition is None else invalid_category_condition | condition

    invalid_binary_condition = None
    for column in BINARY_COLUMNS:
        condition = F.col(column).isNull() | ~F.col(column).isin([0, 1])
        invalid_binary_condition = condition if invalid_binary_condition is None else invalid_binary_condition | condition

    return (
        clean.withColumn("missing_required_value", F.coalesce(missing_required_condition, F.lit(False)))
        .withColumn("invalid_arrival_date", F.col("arrival_date_full").isNull())
        .withColumn("zero_or_negative_night_stay", F.col("total_nights_check") <= 0)
        .withColumn("zero_or_negative_guest_count", F.col("total_guests_check") <= 0)
        .withColumn("negative_numeric_value", F.coalesce(negative_numeric_condition, F.lit(False)))
        .withColumn("invalid_binary_value", F.coalesce(invalid_binary_condition, F.lit(False)))
        .withColumn("invalid_category_value", F.coalesce(invalid_category_condition, F.lit(False)))
        .withColumn(
            "invalid_target_value",
            F.col(TARGET_COLUMN).isNull() | ~F.col(TARGET_COLUMN).isin(VALID_CATEGORY_VALUES[TARGET_COLUMN]),
        )
    )


def invalid_record_condition() -> Column:
    return (
        F.col("missing_required_value")
        | F.col("invalid_arrival_date")
        | F.col("zero_or_negative_night_stay")
        | F.col("zero_or_negative_guest_count")
        | F.col("negative_numeric_value")
        | F.col("invalid_binary_value")
        | F.col("invalid_category_value")
        | F.col("invalid_target_value")
    )


def analyze_dataset(df: DataFrame, title: str) -> None:
    row_count = df.count()

    print(f"\n{title}")
    print(f"Dataset shape: {row_count:,} rows x {len(df.columns)} columns")
    print(f"Target column: {TARGET_COLUMN}")

    print("\nColumn audit:")
    df.printSchema()

    print("\nMissing values:")
    missing_exprs = [
        F.sum(F.when(F.col(column).isNull(), 1).otherwise(0)).alias(column)
        for column in df.columns
    ]
    df.select(missing_exprs).show(truncate=False)

    print("\nTarget distribution:")
    (
        df.groupBy(TARGET_COLUMN)
        .agg(F.count("*").alias("count"))
        .withColumn("percent", F.round(F.col("count") / F.lit(row_count) * 100, 2))
        .orderBy(F.desc("count"))
        .show(truncate=False)
    )

    print("\nCategorical distributions:")
    for column in CATEGORICAL_COLUMNS:
        print(f"\n{column}")
        df.groupBy(column).count().orderBy(F.desc("count")).show(truncate=False)

    print("\nNumeric summary:")
    df.select(NUMERIC_COLUMNS).summary("count", "mean", "stddev", "min", "25%", "50%", "75%", "max").show(truncate=False)


def print_quality_findings(df: DataFrame) -> None:
    checked = add_quality_check_columns(df)
    duplicate_rows = df.count() - df.distinct().count()
    duplicate_booking_ids = df.count() - df.select("booking_id").distinct().count()

    print("\nData quality findings:")
    print("Duplicate full rows:", duplicate_rows)
    print("Duplicate booking_id values:", duplicate_booking_ids)

    checked.select(
        F.sum(F.col("invalid_arrival_date").cast("int")).alias("invalid_arrival_dates"),
        F.sum(F.col("zero_or_negative_night_stay").cast("int")).alias("zero_or_negative_night_stays"),
        F.sum(F.col("zero_or_negative_guest_count").cast("int")).alias("zero_or_negative_guest_count"),
        F.sum((F.col("avg_price_per_room") <= 0).cast("int")).alias("non_positive_room_price"),
    ).show(truncate=False)

    print("Unknown category labels:")
    for column, valid_values in VALID_CATEGORY_VALUES.items():
        unknown = (
            df.select(column)
            .where(~F.col(column).isin(valid_values))
            .distinct()
            .orderBy(column)
            .collect()
        )
        print(f"{column}: {[row[column] for row in unknown]}")


def preprocess_dataset(df: DataFrame) -> tuple[DataFrame, DataFrame]:
    validate_schema(df)

    clean = standardize_text_columns(df)
    clean = coerce_numeric_columns(clean)
    clean = add_quality_check_columns(clean)

    rejected = clean.where(invalid_record_condition()).select(
        "booking_id",
        F.col("missing_required_value").cast("int"),
        F.col("invalid_arrival_date").cast("int"),
        F.col("zero_or_negative_night_stay").cast("int"),
        F.col("zero_or_negative_guest_count").cast("int"),
        F.col("negative_numeric_value").cast("int"),
        F.col("invalid_binary_value").cast("int"),
        F.col("invalid_category_value").cast("int"),
        F.col("invalid_target_value").cast("int"),
    )

    final_columns = [
        "booking_id",
        "no_of_adults",
        "no_of_children",
        "no_of_weekend_nights",
        "no_of_week_nights",
        "type_of_meal_plan",
        "required_car_parking_space",
        "room_type_reserved",
        "lead_time",
        "arrival_year",
        "arrival_month",
        "arrival_date",
        "arrival_date_full",
        "market_segment_type",
        "repeated_guest",
        "no_of_previous_cancellations",
        "no_of_previous_bookings_not_canceled",
        "avg_price_per_room",
        "no_of_special_requests",
        "booking_status",
    ]

    clean = (
        clean.where(~invalid_record_condition())
        .select(final_columns)
        .dropDuplicates(["booking_id"])
    )

    return clean, rejected


def validate_clean_dataset(clean: DataFrame) -> None:
    missing_count = clean.select(
        [
            F.sum(F.when(F.col(column).isNull(), 1).otherwise(0)).alias(column)
            for column in clean.columns
        ]
    ).collect()[0].asDict()

    missing_columns = {column: count for column, count in missing_count.items() if count}

    if missing_columns:
        raise ValueError(f"Clean dataset still has missing values: {missing_columns}")

    duplicate_booking_ids = clean.count() - clean.select("booking_id").distinct().count()

    if duplicate_booking_ids:
        raise ValueError(f"Clean dataset contains {duplicate_booking_ids} duplicate booking_id values.")

    quality = add_quality_check_columns(clean.drop("arrival_date_full"))
    invalid_count = quality.where(invalid_record_condition()).count()

    if invalid_count:
        raise ValueError(f"Clean dataset still has {invalid_count} invalid rows.")

In [6]:
def sanitize_column_suffix(value: str) -> str:
    return re.sub(r"_+", "_", re.sub(r"[^A-Za-z0-9]+", "_", value)).strip("_")


def add_engineered_features(clean: DataFrame) -> DataFrame:
    previous_total_bookings = (
        F.col("no_of_previous_cancellations")
        + F.col("no_of_previous_bookings_not_canceled")
    )

    return (
        clean.withColumn("total_nights", F.col("no_of_weekend_nights") + F.col("no_of_week_nights"))
        .withColumn("total_guests", F.col("no_of_adults") + F.col("no_of_children"))
        .withColumn("has_children", (F.col("no_of_children") > 0).cast("int"))
        .withColumn("arrival_day_of_week", F.dayofweek(F.col("arrival_date_full")))
        .withColumn("arrival_is_weekend", F.col("arrival_day_of_week").isin([1, 7]).cast("int"))
        .withColumn("arrival_quarter", F.quarter(F.col("arrival_date_full")))
        .withColumn(
            "estimated_booking_value",
            F.round(F.col("avg_price_per_room") * F.col("total_nights"), 2),
        )
        .withColumn(
            "avg_price_per_guest",
            F.round(F.col("avg_price_per_room") / F.col("total_guests"), 2),
        )
        .withColumn("previous_total_bookings", previous_total_bookings)
        .withColumn(
            "previous_cancellation_rate",
            F.when(
                previous_total_bookings > 0,
                F.col("no_of_previous_cancellations") / previous_total_bookings,
            ).otherwise(F.lit(0.0)),
        )
        .withColumn(LABEL_COLUMN, F.when(F.col(TARGET_COLUMN) == "Canceled", 1).otherwise(0))
    )


def add_one_hot_features(df: DataFrame) -> tuple[DataFrame, list[str]]:
    encoded = df
    encoded_columns = []

    for column in CATEGORICAL_FEATURE_COLUMNS:
        baseline = ONE_HOT_BASELINES[column]

        for value in VALID_CATEGORY_VALUES[column]:
            if value == baseline:
                continue

            encoded_column = f"{column}_{sanitize_column_suffix(value)}"
            encoded = encoded.withColumn(encoded_column, (F.col(column) == value).cast("int"))
            encoded_columns.append(encoded_column)

    return encoded, encoded_columns


def build_feature_dataset(clean: DataFrame) -> tuple[DataFrame, list[str], list[str]]:
    engineered = add_engineered_features(clean)
    encoded, encoded_columns = add_one_hot_features(engineered)

    feature_columns = SCALABLE_FEATURE_COLUMNS + BINARY_FEATURE_COLUMNS + encoded_columns
    selected_columns = ["booking_id"] + feature_columns + [LABEL_COLUMN]

    return encoded.select(selected_columns), feature_columns, encoded_columns


def print_correlation_analysis(feature_df: DataFrame, feature_columns: list[str]) -> None:
    correlation_columns = feature_columns + [LABEL_COLUMN]

    vector_df = VectorAssembler(
        inputCols=correlation_columns,
        outputCol="correlation_features",
        handleInvalid="skip",
    ).transform(feature_df.select(correlation_columns))

    matrix = Correlation.corr(vector_df, "correlation_features", "pearson").head()[0]
    label_index = len(correlation_columns) - 1

    target_correlations = []

    for index, column in enumerate(feature_columns):
        value = float(matrix[index, label_index])
        if not math.isnan(value):
            target_correlations.append((column, value, abs(value)))

    target_correlations.sort(key=lambda item: item[2], reverse=True)

    print("\nCORRELATION CHECK")
    print("Top features correlated with booking cancellation label:")

    for column, correlation, _ in target_correlations[:12]:
        print(f"{column}: {correlation:.4f}")

    high_feature_pairs = []

    for left_index, left_column in enumerate(feature_columns):
        for right_index in range(left_index + 1, len(feature_columns)):
            right_column = feature_columns[right_index]
            value = float(matrix[left_index, right_index])

            if not math.isnan(value):
                high_feature_pairs.append((left_column, right_column, value, abs(value)))

    high_feature_pairs.sort(key=lambda item: item[3], reverse=True)

    print("\nHighest feature-to-feature correlations:")

    for left_column, right_column, correlation, _ in high_feature_pairs[:10]:
        print(f"{left_column} vs {right_column}: {correlation:.4f}")

In [7]:
def split_train_test(feature_df: DataFrame) -> tuple[DataFrame, DataFrame]:
    split_df = feature_df.withColumn(
        "_split_bucket",
        F.pmod(F.xxhash64(F.col("booking_id")), F.lit(100)),
    )

    train_df = split_df.where(F.col("_split_bucket") < 80).drop("_split_bucket")
    test_df = split_df.where(F.col("_split_bucket") >= 80).drop("_split_bucket")

    return train_df, test_df


def calculate_train_scaling_stats(train_df: DataFrame) -> dict[str, tuple[float, float]]:
    aggregate_expressions = []

    for column in SCALABLE_FEATURE_COLUMNS:
        aggregate_expressions.append(F.mean(F.col(column)).alias(f"{column}_mean"))
        aggregate_expressions.append(F.stddev_samp(F.col(column)).alias(f"{column}_std"))

    stats = train_df.select(aggregate_expressions).collect()[0].asDict()

    return {
        column: (
            float(stats[f"{column}_mean"]),
            float(stats[f"{column}_std"] or 0.0),
        )
        for column in SCALABLE_FEATURE_COLUMNS
    }


def apply_standardization(
    df: DataFrame,
    scaling_stats: dict[str, tuple[float, float]],
    encoded_columns: list[str],
) -> DataFrame:

    scaled_exprs = []

    for column in SCALABLE_FEATURE_COLUMNS:
        mean_value, std_value = scaling_stats[column]
        scaled_column = f"{column}_scaled"

        if std_value == 0:
            scaled_exprs.append(F.lit(0.0).alias(scaled_column))
        else:
            scaled_exprs.append(
                F.round(
                    (F.col(column) - F.lit(mean_value)) / F.lit(std_value),
                    6
                ).alias(scaled_column)
            )

    final_columns = (
        [F.col("booking_id")]
        + scaled_exprs
        + [F.col(c) for c in BINARY_FEATURE_COLUMNS]
        + [F.col(c) for c in encoded_columns]
        + [F.col(LABEL_COLUMN)]
    )

    return df.select(*final_columns)

from pyspark.storagelevel import StorageLevel

def prepare_train_test_datasets(
    feature_df: DataFrame,
    encoded_columns: list[str],
) -> tuple[DataFrame, DataFrame]:

    train_df, test_df = split_train_test(feature_df)

    train_df = train_df.persist(StorageLevel.MEMORY_AND_DISK)
    test_df = test_df.persist(StorageLevel.MEMORY_AND_DISK)

    train_df.count()
    test_df.count()

    scaling_stats = calculate_train_scaling_stats(train_df)

    train_normalized = apply_standardization(train_df, scaling_stats, encoded_columns)
    test_normalized = apply_standardization(test_df, scaling_stats, encoded_columns)

    return train_normalized, test_normalized

In [8]:
raw = (
    spark.sql("SELECT * FROM hotel_db.hotel_reservations_raw")
    .where(F.col("booking_id") != "booking_id")
    .select(EXPECTED_COLUMNS)
)

raw = raw.select([F.col(c).cast("string").alias(c) for c in EXPECTED_COLUMNS])

print("Raw dataset")
raw.printSchema()
raw.show(5, truncate=False)
print("Raw rows:", raw.count())

26/06/09 12:29:21 WARN HiveConf: HiveConf of name hive.stats.jdbc.timeout does not exist
26/06/09 12:29:21 WARN HiveConf: HiveConf of name hive.stats.retries.wait does not exist
26/06/09 12:29:22 WARN ObjectStore: Failed to get database global_temp, returning NoSuchObjectException


Raw dataset
root
 |-- booking_id: string (nullable = true)
 |-- no_of_adults: string (nullable = true)
 |-- no_of_children: string (nullable = true)
 |-- no_of_weekend_nights: string (nullable = true)
 |-- no_of_week_nights: string (nullable = true)
 |-- type_of_meal_plan: string (nullable = true)
 |-- required_car_parking_space: string (nullable = true)
 |-- room_type_reserved: string (nullable = true)
 |-- lead_time: string (nullable = true)
 |-- arrival_year: string (nullable = true)
 |-- arrival_month: string (nullable = true)
 |-- arrival_date: string (nullable = true)
 |-- market_segment_type: string (nullable = true)
 |-- repeated_guest: string (nullable = true)
 |-- no_of_previous_cancellations: string (nullable = true)
 |-- no_of_previous_bookings_not_canceled: string (nullable = true)
 |-- avg_price_per_room: string (nullable = true)
 |-- no_of_special_requests: string (nullable = true)
 |-- booking_status: string (nullable = true)



26/06/09 12:29:24 WARN SessionState: METASTORE_FILTER_HOOK will be ignored, since hive.security.authorization.manager is set to instance of HiveAuthorizerFactory.


+----------+------------+--------------+--------------------+-----------------+-----------------+--------------------------+------------------+---------+------------+-------------+------------+-------------------+--------------+----------------------------+------------------------------------+------------------+----------------------+--------------+
|booking_id|no_of_adults|no_of_children|no_of_weekend_nights|no_of_week_nights|type_of_meal_plan|required_car_parking_space|room_type_reserved|lead_time|arrival_year|arrival_month|arrival_date|market_segment_type|repeated_guest|no_of_previous_cancellations|no_of_previous_bookings_not_canceled|avg_price_per_room|no_of_special_requests|booking_status|
+----------+------------+--------------+--------------------+-----------------+-----------------+--------------------------+------------------+---------+------------+-------------+------------+-------------------+--------------+----------------------------+------------------------------------+--

In [9]:
validate_schema(raw)

raw_for_analysis = coerce_numeric_columns(standardize_text_columns(raw))

analyze_dataset(raw_for_analysis, "DATA UNDERSTANDING")
print_quality_findings(raw_for_analysis)




DATA UNDERSTANDING
Dataset shape: 36,276 rows x 19 columns
Target column: booking_status

Column audit:
root
 |-- booking_id: string (nullable = true)
 |-- no_of_adults: integer (nullable = true)
 |-- no_of_children: integer (nullable = true)
 |-- no_of_weekend_nights: integer (nullable = true)
 |-- no_of_week_nights: integer (nullable = true)
 |-- type_of_meal_plan: string (nullable = true)
 |-- required_car_parking_space: integer (nullable = true)
 |-- room_type_reserved: string (nullable = true)
 |-- lead_time: integer (nullable = true)
 |-- arrival_year: integer (nullable = true)
 |-- arrival_month: integer (nullable = true)
 |-- arrival_date: integer (nullable = true)
 |-- market_segment_type: string (nullable = true)
 |-- repeated_guest: integer (nullable = true)
 |-- no_of_previous_cancellations: integer (nullable = true)
 |-- no_of_previous_bookings_not_canceled: integer (nullable = true)
 |-- avg_price_per_room: double (nullable = true)
 |-- no_of_special_requests: integer (n

26/06/09 12:30:08 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

+-------+------------------+-------------------+--------------------+------------------+--------------------------+-----------------+------------------+------------------+------------------+-------------------+----------------------------+------------------------------------+----------------------+------------------+
|summary|no_of_adults      |no_of_children     |no_of_weekend_nights|no_of_week_nights |required_car_parking_space|lead_time        |arrival_year      |arrival_month     |arrival_date      |repeated_guest     |no_of_previous_cancellations|no_of_previous_bookings_not_canceled|no_of_special_requests|avg_price_per_room|
+-------+------------------+-------------------+--------------------+------------------+--------------------------+-----------------+------------------+------------------+------------------+-------------------+----------------------------+------------------------------------+----------------------+------------------+
|count  |36275             |36275          

In [10]:
clean, rejected = preprocess_dataset(raw)

print("\nPREPROCESSING RESULT")
print("Clean rows:", clean.count())
print("Rejected rows:", rejected.count())

clean.show(5, truncate=False)


PREPROCESSING RESULT
Clean rows: 36160
Rejected rows: 116


[Stage 70:>                                                         (0 + 1) / 1]

+----------+------------+--------------+--------------------+-----------------+-----------------+--------------------------+------------------+---------+------------+-------------+------------+-----------------+-------------------+--------------+----------------------------+------------------------------------+------------------+----------------------+--------------+
|booking_id|no_of_adults|no_of_children|no_of_weekend_nights|no_of_week_nights|type_of_meal_plan|required_car_parking_space|room_type_reserved|lead_time|arrival_year|arrival_month|arrival_date|arrival_date_full|market_segment_type|repeated_guest|no_of_previous_cancellations|no_of_previous_bookings_not_canceled|avg_price_per_room|no_of_special_requests|booking_status|
+----------+------------+--------------+--------------------+-----------------+-----------------+--------------------------+------------------+---------+------------+-------------+------------+-----------------+-------------------+--------------+--------------

In [11]:
feature_df, feature_columns, encoded_columns = build_feature_dataset(clean)

print("Feature columns:", len(feature_columns))
print("Encoded columns:", encoded_columns)

feature_df.show(5, truncate=False)

train_normalized, test_normalized = prepare_train_test_datasets(feature_df, encoded_columns)

print("\nTRAIN TEST SPLIT")
print("Train rows:", train_normalized.count())
print("Test rows:", test_normalized.count())

train_normalized.show(5, truncate=False)

Feature columns: 37
Encoded columns: ['type_of_meal_plan_Meal_Plan_2', 'type_of_meal_plan_Meal_Plan_3', 'type_of_meal_plan_Not_Selected', 'room_type_reserved_Room_Type_2', 'room_type_reserved_Room_Type_3', 'room_type_reserved_Room_Type_4', 'room_type_reserved_Room_Type_5', 'room_type_reserved_Room_Type_6', 'room_type_reserved_Room_Type_7', 'market_segment_type_Aviation', 'market_segment_type_Complementary', 'market_segment_type_Corporate', 'market_segment_type_Offline']
+----------+------------+--------------+--------------------+-----------------+---------+------------+-------------+------------+----------------------------+------------------------------------+------------------+----------------------+------------+------------+-------------------+---------------+-----------------------+-------------------+-----------------------+--------------------------+--------------------------+--------------+------------+------------------+-----------------------------+---------------------------

In [12]:
clean.write.mode("overwrite").saveAsTable("hotel_db.hotel_clean")
rejected.write.mode("overwrite").saveAsTable("hotel_db.hotel_rejected")
train_normalized.write.mode("overwrite").saveAsTable("hotel_db.hotel_train_normalized")
test_normalized.write.mode("overwrite").saveAsTable("hotel_db.hotel_test_normalized")

spark.sql("SHOW TABLES IN hotel_db").show()
spark.sql("SELECT COUNT(*) AS clean_rows FROM hotel_db.hotel_clean").show()
spark.sql("SELECT COUNT(*) AS train_rows FROM hotel_db.hotel_train_normalized").show()
spark.sql("SELECT COUNT(*) AS test_rows FROM hotel_db.hotel_test_normalized").show()

26/06/09 12:30:37 WARN HiveConf: HiveConf of name hive.internal.ss.authz.settings.applied.marker does not exist
26/06/09 12:30:37 WARN HiveConf: HiveConf of name hive.stats.jdbc.timeout does not exist
26/06/09 12:30:37 WARN HiveConf: HiveConf of name hive.stats.retries.wait does not exist
                                                                                

+---------+--------------------+-----------+
|namespace|           tableName|isTemporary|
+---------+--------------------+-----------+
| hotel_db|         hotel_clean|      false|
| hotel_db|      hotel_rejected|      false|
| hotel_db|hotel_reservation...|      false|
| hotel_db|hotel_test_normal...|      false|
| hotel_db|hotel_train_norma...|      false|
+---------+--------------------+-----------+

+----------+
|clean_rows|
+----------+
|     36160|
+----------+

+----------+
|train_rows|
+----------+
|     29027|
+----------+

+---------+
|test_rows|
+---------+
|     7133|
+---------+



In [13]:
from pyspark.storagelevel import StorageLevel
from pyspark.ml.feature import VectorAssembler
from pyspark.sql import functions as F

spark.conf.set("spark.sql.shuffle.partitions", "4")
spark.conf.set("spark.sql.adaptive.enabled", "true")

label_col = LABEL_COLUMN

ml_feature_columns = [
    col for col in train_normalized.columns
    if col not in ["booking_id", label_col]
]

print("ML feature count:", len(ml_feature_columns))

assembler = VectorAssembler(
    inputCols=ml_feature_columns,
    outputCol="features",
    handleInvalid="skip"
)

train_ml = (
    assembler.transform(train_normalized)
    .select("features", F.col(label_col).alias("label"))
    .repartition(4)
    .persist(StorageLevel.MEMORY_AND_DISK)
)

test_ml = (
    assembler.transform(test_normalized)
    .select("features", F.col(label_col).alias("label"))
    .repartition(4)
    .persist(StorageLevel.MEMORY_AND_DISK)
)

# Materialize cache once
train_rows = train_ml.count()
test_rows = test_ml.count()

print("Train ML rows:", train_rows)
print("Test ML rows:", test_rows)
print("Train ML partitions:", train_ml.rdd.getNumPartitions())
print("Test ML partitions:", test_ml.rdd.getNumPartitions())

ML feature count: 37
Train ML rows: 29027
Test ML rows: 7133
Train ML partitions: 4
Test ML partitions: 4


In [16]:
import time

from pyspark.ml.classification import (
    LogisticRegression,
    DecisionTreeClassifier,
    RandomForestClassifier,
    GBTClassifier
)
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.sql import functions as F
from pyspark.storagelevel import StorageLevel

# =====================================================
# MODELLING CONFIG
# =====================================================

RANDOM_SEED = 42

spark.conf.set("spark.sql.shuffle.partitions", "4")
spark.conf.set("spark.sql.adaptive.enabled", "true")

print("Training rows:", train_ml.count())
print("Testing rows:", test_ml.count())
print("Feature count:", len(ml_feature_columns))

# =====================================================
# CLASS DISTRIBUTION CHECK
# =====================================================

print("Training label distribution:")
train_ml.groupBy("label") \
    .count() \
    .withColumn("percentage", F.round(F.col("count") / train_ml.count() * 100, 2)) \
    .orderBy("label") \
    .show()

# =====================================================
# BASELINE ACCURACY
# =====================================================

majority_label = (
    train_ml.groupBy("label")
    .count()
    .orderBy(F.desc("count"))
    .first()["label"]
)

baseline_accuracy = (
    test_ml.withColumn("prediction", F.lit(float(majority_label)))
    .filter(F.col("label") == F.col("prediction"))
    .count() / test_ml.count()
)

print("Baseline majority-class accuracy:", round(baseline_accuracy, 4))

# =====================================================
# MODEL DEFINITIONS
# =====================================================

models_to_train = {
    "Decision Tree": DecisionTreeClassifier(
        featuresCol="features",
        labelCol="label",
        maxDepth=6,
        minInstancesPerNode=20,
        impurity="gini",
        seed=RANDOM_SEED
    ),

    "Random Forest": RandomForestClassifier(
        featuresCol="features",
        labelCol="label",
        numTrees=15,
        maxDepth=6,
        minInstancesPerNode=20,
        subsamplingRate=0.8,
        featureSubsetStrategy="sqrt",
        impurity="gini",
        seed=RANDOM_SEED
    ),

    "Gradient Boosted Tree": GBTClassifier(
        featuresCol="features",
        labelCol="label",
        maxIter=15,
        maxDepth=4,
        stepSize=0.08,
        minInstancesPerNode=20,
        subsamplingRate=0.8,
        seed=RANDOM_SEED
    ),

    "Logistic Regression": LogisticRegression(
        featuresCol="features",
        labelCol="label",
        maxIter=8,
        regParam=0.05,
        elasticNetParam=0.1,
        standardization=False
    )
}

# =====================================================
# EVALUATORS
# =====================================================

auc_evaluator = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

pr_evaluator = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderPR"
)

accuracy_evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)

f1_evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="f1"
)

precision_evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="weightedPrecision"
)

recall_evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="weightedRecall"
)

# =====================================================
# TRAINING + EVALUATION
# =====================================================

trained_models = {}
prediction_cache = {}
results = []

for model_name, estimator in models_to_train.items():
    print(f"\nTraining {model_name}...")

    start_train = time.time()
    model = estimator.fit(train_ml)
    train_time = round(time.time() - start_train, 2)

    trained_models[model_name] = model

    print(f"{model_name} training completed in {train_time} seconds")
    print(f"Evaluating {model_name}...")

    predictions = (
        model.transform(test_ml)
        .select("label", "prediction", "rawPrediction")
        .persist(StorageLevel.MEMORY_AND_DISK)
    )

    predictions.count()

    auc = auc_evaluator.evaluate(predictions)
    pr_auc = pr_evaluator.evaluate(predictions)
    accuracy = accuracy_evaluator.evaluate(predictions)
    precision = precision_evaluator.evaluate(predictions)
    recall = recall_evaluator.evaluate(predictions)
    f1 = f1_evaluator.evaluate(predictions)

    results.append((
        model_name,
        round(auc, 4),
        round(pr_auc, 4),
        round(accuracy, 4),
        round(precision, 4),
        round(recall, 4),
        round(f1, 4),
        train_time
    ))

    prediction_cache[model_name] = predictions

# =====================================================
# PERFORMANCE SUMMARY
# =====================================================

results_df = spark.createDataFrame(
    results,
    [
        "model",
        "roc_auc",
        "pr_auc",
        "accuracy",
        "precision",
        "recall",
        "f1_score",
        "training_time_seconds"
    ]
)

print("\nModel Performance Summary:")
results_df.orderBy(F.desc("roc_auc")).show(truncate=False)

# =====================================================
# CONFUSION MATRIX
# =====================================================

for model_name, predictions in prediction_cache.items():
    print(f"\n{model_name} Confusion Matrix")

    predictions.groupBy("label", "prediction") \
        .count() \
        .orderBy("label", "prediction") \
        .show()

# =====================================================
# TOP FEATURE IMPORTANCE
# =====================================================

for model_name, model in trained_models.items():
    if hasattr(model, "featureImportances"):
        importance_rows = []

        for idx, score in enumerate(model.featureImportances):
            if float(score) > 0:
                importance_rows.append((ml_feature_columns[idx], float(score)))

        importance_rows = sorted(
            importance_rows,
            key=lambda x: x[1],
            reverse=True
        )[:10]

        print(f"\nTop 10 Important Features - {model_name}")

        if importance_rows:
            spark.createDataFrame(
                importance_rows,
                ["feature", "importance"]
            ).show(truncate=False)
        else:
            print("No non-zero feature importance found.")

# =====================================================
# SELECT AND SAVE BEST MODEL
# =====================================================

best_model_name = results_df.orderBy(F.desc("roc_auc")).first()["model"]
best_model = trained_models[best_model_name]

model_save_path = "hdfs:///user/hotel_prediction/saved_models/best_hotel_cancellation_model"

best_model.write().overwrite().save(model_save_path)

print("\nBest model selected:", best_model_name)
print("Best model saved to:", model_save_path)

# =====================================================
# CLEANUP
# =====================================================

for predictions in prediction_cache.values():
    predictions.unpersist()

Training rows: 29027
Testing rows: 7133
Feature count: 37
Training label distribution:
+-----+-----+----------+
|label|count|percentage|
+-----+-----+----------+
|    0|19508|     67.21|
|    1| 9519|     32.79|
+-----+-----+----------+

Baseline majority-class accuracy: 0.6696

Training Decision Tree...
Decision Tree training completed in 0.54 seconds
Evaluating Decision Tree...

Training Random Forest...
Random Forest training completed in 0.67 seconds
Evaluating Random Forest...

Training Gradient Boosted Tree...
Gradient Boosted Tree training completed in 2.45 seconds
Evaluating Gradient Boosted Tree...

Training Logistic Regression...
Logistic Regression training completed in 0.38 seconds
Evaluating Logistic Regression...

Model Performance Summary:
+---------------------+-------+------+--------+---------+------+--------+---------------------+
|model                |roc_auc|pr_auc|accuracy|precision|recall|f1_score|training_time_seconds|
+---------------------+-------+------+-----